# Baseline AMD vs Non-AMD Transfer-Learning Model

This notebook establishes the first binary classification model for the AMD Digital Twin project using an ImageNet-pretrained EfficientNetB0. It defines the data loaders, frozen transfer-learning model, loss, metrics, validation monitoring, and best-checkpoint logic.

The notebook intentionally performs architecture and one-batch validation only. Full training remains opt-in through `RUN_TRAINING = False`.

## 1. Baseline design decisions

- **Architecture:** EfficientNetB0 with ImageNet pretrained weights.
- **Input:** `(3, 224, 224)` tensors from the validated preprocessing pipeline.
- **Head:** dropout followed by one logit for AMD vs Non-AMD.
- **Loss:** `BCEWithLogitsLoss`, using a positive-class weight computed from the training split.
- **Imbalance handling:** the loss weights the minority class; metrics include accuracy, precision, recall, F1, and ROC-AUC.
- **Training policy:** all pretrained base parameters are initially frozen.
- **Checkpoint policy:** the best validation ROC-AUC is saved to `models/baseline_efficientnet_b0_best.pt`.

The existing preprocessing module is reused so image loading stays dynamic and raw images are never copied or modified.

## 2. Imports, reproducibility, and paths

This notebook uses the project `.venv` interpreter, which contains PyTorch, TorchVision, and scikit-learn.

In [1]:
from __future__ import annotations

import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw" / "OCTDL").is_dir():
            return candidate
    raise FileNotFoundError("Could not find a project root containing data/raw/OCTDL.")

PROJECT_ROOT = find_project_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from oct_preprocessing import preprocess_image

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
MODEL_ROOT = PROJECT_ROOT / "models"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = MODEL_ROOT / "baseline_efficientnet_b0_best.pt"
RANDOM_SEED = 42
BATCH_SIZE = 16
NUM_WORKERS = 0
LEARNING_RATE = 1e-3
NUM_EPOCHS = 10
RUN_TRAINING = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")

Project root: C:\Users\vibhu\OneDrive\图片\文档\AMD-Digital-Twin
Device: cpu
Checkpoint path: C:\Users\vibhu\OneDrive\图片\文档\AMD-Digital-Twin\models\baseline_efficientnet_b0_best.pt


## 3. Dynamic OCT dataset and data loaders

Images are loaded from their manifest paths when a sample is requested. Training augmentation is controlled by the existing preprocessing module; validation and test loaders use deterministic preprocessing.

In [2]:
class OCTDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, augment: bool = False, seed: int = RANDOM_SEED):
        self.frame = frame.reset_index(drop=True).copy()
        self.augment = augment
        self.seed = seed

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        record = self.frame.iloc[index]
        rng = np.random.default_rng(self.seed + index) if self.augment else None
        array = preprocess_image(
            PROJECT_ROOT / str(record["image_path"]),
            augment=self.augment,
            rng=rng,
        )
        image = torch.from_numpy(array).float()
        label = torch.tensor(float(record["target_amd"]), dtype=torch.float32)
        return image, label

split_frames = {}
datasets = {}
loaders = {}
for split_name in ["train", "validation", "test"]:
    split_path = PROCESSED_ROOT / f"{split_name}.csv"
    split_frames[split_name] = pd.read_csv(split_path)
    datasets[split_name] = OCTDataset(split_frames[split_name], augment=split_name == "train")
    loaders[split_name] = DataLoader(
        datasets[split_name],
        batch_size=BATCH_SIZE,
        shuffle=split_name == "train",
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    print(f"{split_name}: {len(datasets[split_name]):,} images")

train: 1,428 images
validation: 327 images
test: 307 images


## 4. Binary labels and class imbalance

The positive-class weight is `negative_count / positive_count`. It is applied only to the training loss and does not alter labels or evaluation metrics.

In [3]:
train_labels = split_frames["train"]["target_amd"].astype(int)
positive_count = int((train_labels == 1).sum())
negative_count = int((train_labels == 0).sum())
POSITIVE_CLASS_WEIGHT = negative_count / positive_count

print(f"Training AMD images: {positive_count:,}")
print(f"Training non-AMD images: {negative_count:,}")
print(f"Positive-class loss weight: {POSITIVE_CLASS_WEIGHT:.4f}")

for split_name, frame in split_frames.items():
    print(f"{split_name} labels: {frame['target_amd'].value_counts().sort_index().to_dict()}")

Training AMD images: 858
Training non-AMD images: 570
Positive-class loss weight: 0.6643
train labels: {0: 570, 1: 858}
validation labels: {0: 130, 1: 197}
test labels: {0: 133, 1: 174}


## 5. Build the frozen EfficientNetB0 baseline

The pretrained convolutional base is frozen initially. The original 1,000-class classifier is replaced with dropout and one output logit.

In [4]:
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)
for parameter in model.features.parameters():
    parameter.requires_grad = False

classifier_input_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.30),
    nn.Linear(classifier_input_features, 1),
)
model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([POSITIVE_CLASS_WEIGHT], device=DEVICE)
)
optimizer = torch.optim.Adam(
    filter(lambda parameter: parameter.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
)

trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
non_trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if not parameter.requires_grad)
print(model)
print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Non-trainable parameters: {non_trainable_parameters:,}")
print(f"Total parameters: {trainable_parameters + non_trainable_parameters:,}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\vibhu/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth



0.6%


1.2%


1.8%


2.4%

...


98.4%


99.0%


99.6%


100.0%

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          

## 6. Required forward-pass validation

Before training, run one batch through the model and confirm input, label, logit, and probability shapes are compatible.

In [5]:
model.eval()
sample_images, sample_labels = next(iter(loaders["train"]))
sample_images = sample_images.to(DEVICE)
sample_labels = sample_labels.to(DEVICE)
with torch.no_grad():
    sample_logits = model(sample_images).squeeze(1)
    sample_probabilities = torch.sigmoid(sample_logits)

print(f"Input tensor shape: {tuple(sample_images.shape)}")
print(f"Label tensor shape: {tuple(sample_labels.shape)}")
print(f"Logit tensor shape: {tuple(sample_logits.shape)}")
print(f"Probability tensor shape: {tuple(sample_probabilities.shape)}")
assert sample_images.ndim == 4 and sample_images.shape[1:] == (3, 224, 224)
assert sample_labels.shape == sample_logits.shape == sample_probabilities.shape
assert torch.isfinite(sample_logits).all()
print("Forward-pass validation passed.")

Input tensor shape: (16, 3, 224, 224)
Label tensor shape: (16,)
Logit tensor shape: (16,)
Probability tensor shape: (16,)
Forward-pass validation passed.


## 7. Evaluation metrics and training functions

These functions are defined for the baseline run. Validation ROC-AUC selects the best checkpoint; the remaining requested metrics are reported at a threshold of 0.5.

In [6]:
def calculate_metrics(labels: list[int], probabilities: list[float]) -> dict[str, float]:
    labels_array = np.asarray(labels, dtype=int)
    probabilities_array = np.asarray(probabilities, dtype=float)
    predictions = (probabilities_array >= 0.5).astype(int)
    metrics = {
        "accuracy": accuracy_score(labels_array, predictions),
        "precision": precision_score(labels_array, predictions, zero_division=0),
        "recall": recall_score(labels_array, predictions, zero_division=0),
        "f1": f1_score(labels_array, predictions, zero_division=0),
    }
    metrics["roc_auc"] = roc_auc_score(labels_array, probabilities_array) if len(np.unique(labels_array)) == 2 else float("nan")
    return metrics

def run_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer=None) -> tuple[float, dict[str, float]]:
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    all_labels, all_probabilities = [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if is_training:
            optimizer.zero_grad(set_to_none=True)
        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)
        if is_training:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * len(labels)
        all_labels.extend(labels.detach().cpu().numpy().astype(int).tolist())
        all_probabilities.extend(torch.sigmoid(logits).detach().cpu().numpy().tolist())
    return total_loss / len(loader.dataset), calculate_metrics(all_labels, all_probabilities)

def train_model(model: nn.Module, loaders: dict, criterion: nn.Module, optimizer, epochs: int) -> pd.DataFrame:
    history = []
    best_validation_auc = -np.inf
    for epoch in range(1, epochs + 1):
        train_loss, train_metrics = run_epoch(model, loaders["train"], criterion, optimizer)
        validation_loss, validation_metrics = run_epoch(model, loaders["validation"], criterion)
        row = {"epoch": epoch, "train_loss": train_loss, "validation_loss": validation_loss}
        row.update({f"train_{key}": value for key, value in train_metrics.items()})
        row.update({f"validation_{key}": value for key, value in validation_metrics.items()})
        history.append(row)
        validation_auc = validation_metrics["roc_auc"]
        if validation_auc > best_validation_auc:
            best_validation_auc = validation_auc
            torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "validation_metrics": validation_metrics}, CHECKPOINT_PATH)
        print(f"Epoch {epoch}: train loss={train_loss:.4f}, validation ROC-AUC={validation_auc:.4f}")
    return pd.DataFrame(history)

## 8. Optional baseline training

Training is intentionally disabled for this first validation pass. Set `RUN_TRAINING = True` only after reviewing the architecture and forward-pass checks. The training function saves the best validation ROC-AUC checkpoint.

In [7]:
if RUN_TRAINING:
    history = train_model(model, loaders, criterion, optimizer, NUM_EPOCHS)
    history.to_csv(PROCESSED_ROOT / "baseline_training_history.csv", index=False)
    print(f"Best checkpoint saved to: {CHECKPOINT_PATH}")
else:
    print("Training skipped. Architecture and forward-pass validation are complete.")

Training skipped. Architecture and forward-pass validation are complete.


## 9. Next step

The next notebook should enable the controlled baseline run, evaluate the untouched test set once, and report the requested metrics with a confusion matrix and ROC curve. Hyperparameter tuning and unfreezing the EfficientNet base should remain separate experiments.